In [1]:
import pandas as pd
import numpy as np

%pip install yfinance

import yfinance as yf

Note: you may need to restart the kernel to use updated packages.


# Primary KPI ETL 
## Cumaltive Return, Annualized Volatility, Maximum Drawdown

In [2]:
#Downloads Market Data from Yahoo! Finance’s API from specific tickers: AMD, Nvidia, Intel, Nasdaq,  
raw_df = yf.download(
    tickers=["AMD", "NVDA", "INTC", "TSM", "AVGO", "SPY"],
    start="2021-01-01",
    end="2026-01-01",
    interval="1d",
    auto_adjust=True,
    group_by="ticker",
)


#Preserves raw dataframe into a comma seperate value file
raw_df.to_csv("raw_market_data_2021_2025.csv")

#Inspects the top rows
raw_df.head()

[*********************100%***********************]  6 of 6 completed


Ticker            AMD                                                    TSM  \
Price            Open       High        Low      Close    Volume        Open   
Date                                                                           
2021-01-04  92.110001  96.059998  90.919998  92.300003  51802600  101.703242   
2021-01-05  92.099998  93.209999  91.410004  92.769997  34208000  102.560884   
2021-01-06  91.620003  92.279999  89.459999  90.330002  51911700  103.874705   
2021-01-07  91.330002  95.510002  91.199997  95.160004  42897200  108.883676   
2021-01-08  95.980003  96.400002  93.269997  94.580002  39816400  114.878042   

Ticker                                                    ...         SPY  \
Price             High         Low       Close    Volume  ...        Open   
Date                                                      ...               
2021-01-04  104.102805  100.772608  101.913086  11262100  ...  347.627801   
2021-01-05  104.878329  102.195930  102.889336  10583600  ...  340.949572   
2021-01-06  106.192156  102.688612  105.480499  10609300  ...  342.440764   
2021-01-07  112.168257  107.287007  110.790558  13556100  ...  348.359610   
2021-01-08  115.224745  106.730469  108.290642  18976800  ...  352.518352   

Ticker                                                          INTC  \
Price             High         Low       Close     Volume       Open   
Date                                                                   
2021-01-04  347.757488  337.911534  341.588715  110210800  45.101826   
2021-01-05  345.025030  340.903243  343.941315   66426200  44.704058   
2021-01-06  349.174558  341.894285  345.997528  107997700  45.599034   
2021-01-07  351.879318  348.183622  351.138336   68766800  46.548265   
2021-01-08  353.351964  349.285777  353.138947   71677200  47.416129   

Ticker                                                 
Price            High        Low      Close    Volume  
Date                                                   
2021-01-04  46.457864  44.658855  44.902939  46102500  
2021-01-05  45.951614  44.595576  45.752728  24866600  
2021-01-06  47.144919  45.382069  46.195690  36809600  
2021-01-07  47.633093  46.376498  47.181080  32092300  
2021-01-08  47.488449  46.349379  46.692909  34625400  

[5 rows x 30 columns]

In [3]:
#Processing a single ticker to ensure there are no issues using amd as the test
amd_df = raw_df["AMD"].copy()

#Drops non-trading days/nulls and move Trading Date out of index into a column
amd_df = amd_df.dropna(subset=["Close"]).reset_index()

#Renames Close to Adj_Close (since auto_adjust=True was set)
amd_df.rename(columns={"Date": "Trading_Date", "Close": "Adj_Close"}, inplace=True)

#Computes Daily Return: (P_t - P_{t-1}) / P_{t-1}
amd_df["Daily_Return"] = amd_df["Adj_Close"].pct_change()

#Computes Cumulative Return: (P_t / P_0) - 1
p0 = amd_df["Adj_Close"].iloc[0]
amd_df["Cumulative_Return"] = (amd_df["Adj_Close"] / p0) - 1

#Computes Rolling Peak and Drawdown
amd_df["Rolling_Peak"] = amd_df["Adj_Close"].cummax()
amd_df["Drawdown"] = (amd_df["Adj_Close"] - amd_df["Rolling_Peak"]) / amd_df["Rolling_Peak"]

#Adds Ticker & Asset Class metadata
amd_df["Ticker"] = "AMD"
amd_df["Asset_Class"] = "Equity"

#Selects final schema columns
cols = [
    "Trading_Date", "Ticker", "Asset_Class", "Adj_Close", 
    "Volume", "Daily_Return", "Cumulative_Return", "Drawdown"
]
amd_clean = amd_df[cols]

#Calculates AMD's overall 3 Key KPIs
amd_cum_return = amd_clean["Cumulative_Return"].iloc[-1]
amd_ann_volatility = amd_clean["Daily_Return"].std() * np.sqrt(252)
amd_max_drawdown = amd_clean["Drawdown"].min()

print("=== AMD Test Transformation Complete ===")
print(f"Cumulative Return (2021-2025): {amd_cum_return * 100:.2f}%")
print(f"Annualized Volatility:         {amd_ann_volatility * 100:.2f}%")
print(f"Maximum Drawdown:              {amd_max_drawdown * 100:.2f}%\n")
amd_clean.head()
#note, the only 0's and NaN's are on the first date recorded since returns are not possible to compare

=== AMD Test Transformation Complete ===
Cumulative Return (2021-2025): 132.03%
Annualized Volatility:         52.41%
Maximum Drawdown:              -65.45%



Price,Trading_Date,Ticker,Asset_Class,Adj_Close,Volume,Daily_Return,Cumulative_Return,Drawdown
0,2021-01-04,AMD,Equity,92.300003,51802600,NaN,0.000000,0.000000
1,2021-01-05,AMD,Equity,92.769997,34208000,0.005092,0.005092,0.000000
2,2021-01-06,AMD,Equity,90.330002,51911700,-0.026302,-0.021343,-0.026302
3,2021-01-07,AMD,Equity,95.160004,42897200,0.053471,0.030986,0.000000
4,2021-01-08,AMD,Equity,94.580002,39816400,-0.006095,0.024702,-0.006095


In [4]:
tickers = ["AMD", "NVDA", "INTC", "TSM", "AVGO", "SPY"]
processed_frames = []

#ETL Loop for all tickers
for ticker in tickers:
    #Extracts ticker slice, drop nulls, reset index
    df_ticker = raw_df[ticker].copy().dropna(subset=["Close"]).reset_index()
    df_ticker.rename(columns={"Date": "Trading_Date", "Close": "Adj_Close"}, inplace=True)
    
    #Calculates performance and risk metrics
    df_ticker["Daily_Return"] = df_ticker["Adj_Close"].pct_change()
    
    p0 = df_ticker["Adj_Close"].iloc[0]
    df_ticker["Cumulative_Return"] = (df_ticker["Adj_Close"] / p0) - 1
    
    df_ticker["Rolling_Peak"] = df_ticker["Adj_Close"].cummax()
    df_ticker["Drawdown"] = (df_ticker["Adj_Close"] - df_ticker["Rolling_Peak"]) / df_ticker["Rolling_Peak"]
    
    #Tags metadata
    df_ticker["Ticker"] = ticker
    df_ticker["Asset_Class"] = "Benchmark" if ticker == "SPY" else "Equity"
    
    #Filters standard long schema
    cols = ["Trading_Date", "Ticker", "Asset_Class", "Adj_Close", "Volume", "Daily_Return", "Cumulative_Return", "Drawdown"]
    processed_frames.append(df_ticker[cols])

#Transforms into long-format dataset
clean_long_df = pd.concat(processed_frames, ignore_index=True)
clean_long_df.sort_values(by=["Trading_Date", "Ticker"], inplace=True)

#Exports clean dataset to CSV for Power BI & Python analysis
clean_long_df.to_csv("cleaned_market_data_2021_2025.csv", index=False)
print("Saved 'cleaned_market_data_2021_2025.csv'.")


Saved 'cleaned_market_data_2021_2025.csv'.


In [5]:
#Generates Overall 3 KPI Summary Table
kpi_summary = clean_long_df.groupby("Ticker").agg(
    Asset_Class=("Asset_Class", "first"),
    Start_Price=("Adj_Close", "first"),
    End_Price=("Adj_Close", "last"),
    Cumulative_Return=("Cumulative_Return", "last"),
    Annualized_Volatility=("Daily_Return", lambda x: x.std() * np.sqrt(252)),
    Max_Drawdown=("Drawdown", "min")
).reset_index()

#Saves unformatted numerical KPI summary
kpi_summary.to_csv("kpi_summary_metrics_2021_2025.csv", index=False)

#Formats for clean visual display as percentages
kpi_summary_disp = kpi_summary.copy()
kpi_summary_disp["Cumulative_Return"] = (kpi_summary_disp["Cumulative_Return"] * 100).round(2).astype(str) + "%"
kpi_summary_disp["Annualized_Volatility"] = (kpi_summary_disp["Annualized_Volatility"] * 100).round(2).astype(str) + "%"
kpi_summary_disp["Max_Drawdown"] = (kpi_summary_disp["Max_Drawdown"] * 100).round(2).astype(str) + "%"

print("\n=== Final KPI Summary Table (2021-2025) ===")
print(kpi_summary_disp.to_string(index=False))


=== Final KPI Summary Table (2021-2025) ===
Ticker Asset_Class  Start_Price  End_Price Cumulative_Return Annualized_Volatility Max_Drawdown
   AMD      Equity    92.300003 214.160004           132.03%                52.41%      -65.45%
  AVGO      Equity    38.043430 344.203003           804.76%                42.31%      -41.15%
  INTC      Equity    44.902939  36.900002           -17.82%                46.24%       -70.8%
  NVDA      Equity    13.046196 186.064606           1326.2%                52.22%      -66.34%
   SPY   Benchmark   341.588715 676.635010            98.08%                17.11%       -24.5%
   TSM      Equity   101.913086 301.519226           195.86%                37.37%      -56.47%


# Supporting Measures
## Trading-Day Count, Normalized Price, Daily Return, Benchmark Difference, Annual Return

In [6]:
#Ensures proper data types and chronological ordering
clean_long_df["Trading_Date"] = pd.to_datetime(clean_long_df["Trading_Date"])
clean_long_df.sort_values(by=["Ticker", "Trading_Date"], inplace=True)

#Trading-Day Count (Sequential index 1..N per ticker)
clean_long_df["Trading_Day_Count"] = clean_long_df.groupby("Ticker").cumcount() + 1

#Normalized Price (Indexed to 100 on start date per ticker)
p0_series = clean_long_df.groupby("Ticker")["Adj_Close"].transform("first")
clean_long_df["Normalized_Price"] = (clean_long_df["Adj_Close"] / p0_series) * 100

#Benchmark Difference (Daily Return minus SPY Daily Return)
spy_returns = clean_long_df[clean_long_df["Ticker"] == "SPY"].set_index("Trading_Date")["Daily_Return"]
clean_long_df["SPY_Daily_Return"] = clean_long_df["Trading_Date"].map(spy_returns)
clean_long_df["Benchmark_Difference"] = clean_long_df["Daily_Return"] - clean_long_df["SPY_Daily_Return"]
clean_long_df.drop(columns=["SPY_Daily_Return"], inplace=True)

#Annual Return (Year-over-Year return within each calendar year)
clean_long_df["Year"] = clean_long_df["Trading_Date"].dt.year
clean_long_df["Annual_Return"] = clean_long_df.groupby(["Ticker", "Year"])["Adj_Close"].transform(
    lambda x: (x.iloc[-1] / x.iloc[0]) - 1
)

#Sorts strictly by date and ticker
clean_long_df.sort_values(by=["Trading_Date", "Ticker"], inplace=True)

#Saves unformatted numerical dataset (Primary CSV for Power BI & Python)
clean_long_df.to_csv("cleaned_market_data_2021_2025.csv", index=False)
print("Updated 'cleaned_market_data_2021_2025.csv' with supporting measures.")

clean_long_df_formatted = clean_long_df.copy()

pct_columns = ["Daily_Return", "Cumulative_Return", "Drawdown", "Benchmark_Difference", "Annual_Return"]
for col in pct_columns:
    clean_long_df_formatted[col] = (clean_long_df_formatted[col] * 100).round(2).astype(str) + "%"

clean_long_df_formatted["Normalized_Price"] = clean_long_df_formatted["Normalized_Price"].round(2)

clean_long_df_formatted.to_csv("cleaned_market_data_formatted_2021_2025.csv", index=False)
print("Saved 'cleaned_market_data_formatted_2021_2025.csv'.")

Updated 'cleaned_market_data_2021_2025.csv' with supporting measures.
Saved 'cleaned_market_data_formatted_2021_2025.csv'.


In [7]:
supporting_measures_disp = clean_long_df.copy()

#Formats supporting measures as percentages
supporting_measures_disp["Daily_Return"] = (supporting_measures_disp["Daily_Return"] * 100).round(2).astype(str) + "%"
supporting_measures_disp["Benchmark_Difference"] = (supporting_measures_disp["Benchmark_Difference"] * 100).round(2).astype(str) + "%"
supporting_measures_disp["Annual_Return"] = (supporting_measures_disp["Annual_Return"] * 100).round(2).astype(str) + "%"
supporting_measures_disp["Normalized_Price"] = supporting_measures_disp["Normalized_Price"].round(2)

print("\n=== Supporting Measures Preview (2021-2025) ===")
cols_to_show = ["Trading_Date", "Ticker", "Trading_Day_Count", "Normalized_Price", "Daily_Return", "Benchmark_Difference", "Annual_Return"]
print(supporting_measures_disp[cols_to_show].head(10).to_string(index=False))
#note, the only 0's and NaN's are on the first date recorded since returns are not possible to compare


=== Supporting Measures Preview (2021-2025) ===
Trading_Date Ticker  Trading_Day_Count  Normalized_Price Daily_Return Benchmark_Difference Annual_Return
  2021-01-04    AMD                  1            100.00         nan%                 nan%         55.9%
  2021-01-04   AVGO                  1            100.00         nan%                 nan%        61.13%
  2021-01-04   INTC                  1            100.00         nan%                 nan%         6.37%
  2021-01-04   NVDA                  1            100.00         nan%                 nan%       124.48%
  2021-01-04    SPY                  1            100.00         nan%                 nan%        30.51%
  2021-01-04    TSM                  1            100.00         nan%                 nan%         9.43%
  2021-01-05    AMD                  2            100.51        0.51%               -0.18%         55.9%
  2021-01-05   AVGO                  2            100.68        0.68%               -0.01%        61.13%
  2021